# PageIndex Multimodal: ingest, retrieve, tools, delete

PageIndex local mode (package `pageindex` asli, tanpa diubah) ditambah penanganan multimodal dari `vectorless_rag`:

- **Halaman bertabel atau bergambar** dikirim ke Textract: tabel menjadi markdown, dan gambar di-crop, di-upload ke S3, lalu dideskripsikan oleh VLM. Halaman lain tetap memakai teks asli. Summary node ikut membaca teks Textract ini.
- **PDF scan dan file `.jpg` / `.png`** dibaca dengan OCR (Textract); tree-nya dari heading Textract.
- **File office** (`.docx`, `.pptx`, ...) dikonversi ke PDF lewat LibreOffice (`unoconvert`), **`.xlsx`** diindeks per sheet.
- Flash, format index, dan retrieval sama persis dengan PageIndex asli; summary dan deskripsi ditulis dalam Bahasa Indonesia.

Konfigurasi (LLM, VLM, Textract, S3) dibaca dari `.env`. Jalankan notebook dari root repo.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(".env")   # config multimodal (config.py) juga membaca .env dari working directory

from vectorless_rag import MultimodalClient, PageIndexAPIError, pageindex_tools

client = MultimodalClient(
    index_model=os.environ["PAGEINDEX_INDEX_MODEL"],
    chat_model=os.environ["PAGEINDEX_CHAT_MODEL"],
    storage_path=".pageindex",
    summary_concurrency=8,   # sesuaikan dengan rate limit proxy LLM
)

## 1. Ingest

`submit_document` bersifat sinkron: fungsi baru kembali setelah dokumen selesai diindeks.

In [ ]:
FILES = [
    "data/Rancangan_Akhir_Renstra_Diskominfo_Jatim_2025-2029.pdf",            # PDF digital + tabel
    "data/Notulen_Rapat_Teknis_Integrasi_Aplikasi_Diskominfo_Jatim_2025.pdf",
    "data/DOC_L1-010_scan_halaman_tengah_renja_jatim.docx",                   # office -> PDF (LibreOffice)
]

docs = {}
for path in FILES:
    doc = client.submit_document(path)
    docs[doc["name"]] = doc["doc_id"]
    print(doc["name"], doc["doc_id"], doc["usage"]["total_tokens"], "token")

In [ ]:
for d in client.list_documents()["documents"]:
    print(d["id"], "|", d["name"], "|", d.get("description"))

## 2. Retrieve (tanpa LLM)

Fungsi-fungsi ini adalah data yang juga dibaca tool agent:
- `get_document_structure`: tree berisi judul, rentang halaman, dan summary.
- `get_page_content`: teks halaman. Untuk halaman yang lewat Textract, isinya markdown dengan tabel dan deskripsi gambar.

In [ ]:
renstra = "Rancangan_Akhir_Renstra_Diskominfo_Jatim_2025-2029.pdf"
doc_id = docs[renstra]

def print_tree(nodes, depth=0):
    for node in nodes:
        print(f"{'  ' * depth}- [{node['start_index']}-{node['end_index']}] {node['title']}")
        print_tree(node.get("nodes") or [], depth + 1)

print_tree(client.get_document_structure(doc_id))

In [ ]:
for page in client.get_page_content(doc_id, "1"):
    print(f"--- halaman {page['page_index']} ---")
    print(page["markdown"][:2000])

## 3. Tools untuk agent

`pageindex_tools(client)` memberi 4 LangGraph tool (`browse_documents`, `get_document`, `get_document_structure`, `get_page_content`) untuk dipasang di agent. `doc_ids` membatasi dokumen yang boleh dibaca. Setiap halaman dari `get_page_content` membawa `citation` (`file` + `page`, atau `file` + `sheet` untuk `.xlsx`).

In [ ]:
tools = {t.name: t for t in pageindex_tools(client)}
print(tools["browse_documents"].invoke({}))

In [ ]:
print(tools["get_document_structure"].invoke({"doc_name": renstra}))

In [ ]:
# Dokumen office (dikonversi ke PDF, halaman scan dibaca lewat OCR)
docx = "DOC_L1-010_scan_halaman_tengah_renja_jatim.docx"
print(tools["get_page_content"].invoke({"doc_name": docx, "pages": "1"}))

In [ ]:
# Dibatasi ke Renstra saja: dokumen lain ditolak
scoped = {t.name: t for t in pageindex_tools(client, doc_ids=[docs[renstra]])}
print(scoped["get_page_content"].invoke({"doc_name": docx, "pages": "1"}))

## 4. Delete

Penghapusan bersifat permanen: folder dokumen dan entry-nya di manifest ikut terhapus.

In [ ]:
print(client.delete_document(doc_id))

try:
    client.get_document(doc_id)
except PageIndexAPIError as e:
    print("get_document ->", e)

In [ ]:
# Hapus semua dokumen di library ini
for d in client.list_documents(limit=1000)["documents"]:
    client.delete_document(d["id"])
print("tersisa:", client.list_documents()["total"])